## EXPLORATORY DATA ANALYSIS

This notebook illustrates how data is structured ,the quality of data and patterns we can recocgnize before moving to ML

In [1]:
import pandas as pd
import numpy as np 
import seaborn as sns 
import os


In [23]:
from pathlib import Path
NOTEBOOK_DIR=Path.cwd()
ROOT_DIR=NOTEBOOK_DIR.parent
DATA_DIR=ROOT_DIR/"data"/"raw"
DATA_DIR_PROCESSED= ROOT_DIR/"data"/"processed"


In [3]:
print("X TRAIN DS")
x_train=pd.read_csv(DATA_DIR/ "x_train.csv" , index_col='ID')
display(x_train.head())
print(f"(nb_rows,nb_columns)={x_train.shape}")

print("Y TRAIN DS")
y_train= pd.read_csv(DATA_DIR/"y_train.csv", index_col="ID")
display(y_train)
print(f"(nb_rows,nb_columns)=f{y_train.shape}")

X TRAIN DS


,DATE,STOCK,INDUSTRY,INDUSTRY_GROUP,SECTOR,SUB_INDUSTRY,RET_1,VOLUME_1,RET_2,VOLUME_2,...,RET_16,VOLUME_16,RET_17,VOLUME_17,RET_18,VOLUME_18,RET_19,VOLUME_19,RET_20,VOLUME_20
ID,,,,,,,,,,,,,,,,,,,,,
0,0,2,18,5,3,44,-0.015748,0.147931,-0.015504,0.179183,...,0.059459,0.630899,0.003254,-0.379412,0.008752,-0.110597,-0.012959,0.174521,-0.002155,-0.000937
1,0,3,43,15,6,104,0.003984,NaN,-0.090580,NaN,...,0.015413,NaN,0.003774,NaN,-0.018518,NaN,-0.028777,NaN,-0.034722,NaN
2,0,4,57,20,8,142,0.000440,-0.096282,-0.058896,0.084771,...,0.008964,-0.010336,-0.017612,-0.354333,-0.006562,-0.519391,-0.012101,-0.356157,-0.006867,-0.308868
3,0,8,1,1,1,2,0.031298,-0.429540,0.007756,-0.089919,...,-0.031769,0.012105,0.033824,-0.290178,-0.001468,-0.663834,-0.013520,-0.562126,-0.036745,-0.631458
4,0,14,36,12,5,92,0.027273,-0.847155,-0.039302,-0.943033,...,-0.038461,-0.277083,-0.012659,0.139086,0.004237,-0.017547,0.004256,0.579510,-0.040817,0.802806


(nb_rows,nb_columns)=(418595, 46)
Y TRAIN DS


,RET
ID,
0,True
1,True
2,False
3,False
4,False
...,...
418590,False
418591,False
418592,True


(nb_rows,nb_columns)=f(418595, 1)


- RET_1...RET_20 : correspond to the return residual of yesterday till the return residual of d-20 days 
_ VOLUME_1...VOLUME_20 : correspond to the relative volume (witout the market trend) of yesterday till the relative volume od d-20days 
- Features like : Sector / industery group / industry and sub industry : should be understood and they're classification from broadest to narrowest in order to know which one to use if we want a narrowest peer group ...
- Dates are annonymised in order to not use them by sorting the data chronoligaclly and using calender effect to infer the day of the weeks effects etc ...

In [4]:
features_with_mv= x_train.isna().sum()[x_train.isna().sum()>0]
display(features_with_mv)
print(f" {len(features_with_mv)} features, with NA values.")

RET_1         2359
VOLUME_1     65025
RET_2         2465
VOLUME_2     66386
RET_3         2507
VOLUME_3     67819
RET_4         2544
VOLUME_4     70997
RET_5         2584
VOLUME_5     74693
RET_6         2597
VOLUME_6     74714
RET_7         2585
VOLUME_7     73853
RET_8         2623
VOLUME_8     73898
RET_9         2682
VOLUME_9     73298
RET_10        2692
VOLUME_10    73305
RET_11        2961
VOLUME_11    72025
RET_12        3186
VOLUME_12    62523
RET_13        3360
VOLUME_13    59008
RET_14        4413
VOLUME_14    60929
RET_15        4990
VOLUME_15    66373
RET_16        5280
VOLUME_16    67262
RET_17        5301
VOLUME_17    62314
RET_18        5307
VOLUME_18    67586
RET_19        5313
VOLUME_19    67329
RET_20        5341
VOLUME_20    67857
dtype: int64

 40 features, with NA values.


- we need to determine the type of missingnes ( MCAR MAR MNAR)

In [5]:
missing_mask= x_train.isna()

# we create a flag for each columns containing a missing values in order to see if there's correlation with other features

for col in features_with_mv.index: # les indexes de la Serie sont RET_1 VOLUME_1 RET_2 ...
    x_train[f"{col}_missing"]=missing_mask[col].astype(int)

# if is_missing column is correlated to Y_train means that MNAR  and is_missing should be a feature itself
cor_col=[]
for col in features_with_mv.index:
    cor= x_train[f"{col}_missing"].corr(y_train.iloc[:,0])
    cor_col.append(cor)
    
corr_df=pd.DataFrame({
    "column_miss" : [f for f in features_with_mv.index],
    "correlation_with_y_train": [ c for c in cor_col]
})
    
corr_df_sorted=corr_df.sort_values(by="correlation_with_y_train", ascending=False).reset_index(drop=True)
corr_df_sorted.head(6)



,column_miss,correlation_with_y_train
0,RET_20,0.005162
1,RET_17,0.004923
2,RET_18,0.004793
3,RET_14,0.004784
4,RET_19,0.004705
5,RET_16,0.004696


- correlation with the y_train is practically meaningless because it's gonna only explain a negligeable fraction of variance, meaning that features aren't MNAR...

- let's check if missingnes depends on other observed feature(Sector,industrygroupe,industry or subindustry) and not the value itself (i.e MAR)

In [6]:
missing_col=[f"{col}_missing" for col in features_with_mv.index]
rates=x_train.groupby('SECTOR')[missing_col].mean()

In [7]:
spread=(rates.max()-rates.min())
spread.head(10).sort_values(ascending=False)

VOLUME_5_missing    0.191630
VOLUME_4_missing    0.182373
VOLUME_3_missing    0.173909
VOLUME_2_missing    0.169985
VOLUME_1_missing    0.165285
RET_3_missing       0.007911
RET_2_missing       0.007752
RET_5_missing       0.007594
RET_4_missing       0.007435
RET_1_missing       0.007277
dtype: float64

In [8]:
missing_cols=[f"{c}_missing" for c in features_with_mv.index]

features=["SECTOR", "INDUSTRY_GROUP", "INDUSTRY", "SUB_INDUSTRY"]

for f in features:
    rates= x_train.groupby(f)[missing_cols].mean()
    print (f"***** Spread grouped by {f}")
    spread= ((rates.max()-rates.min())*100).sort_values(ascending=False)
    print(spread.head(6))

***** Spread grouped by SECTOR
VOLUME_5_missing     19.163037
VOLUME_6_missing     19.104607
VOLUME_8_missing     18.759050
VOLUME_9_missing     18.745212
VOLUME_7_missing     18.717837
VOLUME_10_missing    18.706822
dtype: float64
***** Spread grouped by INDUSTRY_GROUP
VOLUME_5_missing     20.442779
VOLUME_6_missing     20.379836
VOLUME_8_missing     20.282776
VOLUME_7_missing     20.227126
VOLUME_9_missing     20.127177
VOLUME_10_missing    20.107159
dtype: float64
***** Spread grouped by INDUSTRY
VOLUME_5_missing     21.711346
VOLUME_6_missing     21.665221
VOLUME_7_missing     21.233788
VOLUME_8_missing     21.205499
VOLUME_9_missing     21.195208
VOLUME_10_missing    21.130283
dtype: float64
***** Spread grouped by SUB_INDUSTRY
VOLUME_5_missing     31.372549
VOLUME_6_missing     31.372549
VOLUME_8_missing     30.718954
VOLUME_7_missing     30.718954
VOLUME_11_missing    30.501089
VOLUME_10_missing    30.283224
dtype: float64


- missingnes depends on business categories... with sector we have 19% spread which is consistence with MAR 
- but we need to be !!!cautious!!! with saying that sub_industry is is more informative , because if samples size decreases the variability is gonna increase ( it doesn't always mean that the sub group is more informative ) let's see if it's the case

In [9]:
for f in features :
    sizes=x_train.groupby(f).size()
    print(f"{f}: feature, {len(sizes)} : number of unique values ,{sizes.min()}: the minimum value, {sizes.median() }:the median"
    )

SECTOR: feature, 12 : number of unique values ,3054: the minimum value, 20115.5:the median
INDUSTRY_GROUP: feature, 26 : number of unique values ,3054: the minimum value, 14255.5:the median
INDUSTRY: feature, 72 : number of unique values ,56: the minimum value, 3760.5:the median
SUB_INDUSTRY: feature, 175 : number of unique values ,32: the minimum value, 1362.0:the median


- number of groups increases from 12 to 175

- as expected the median size per group drops from 20 115.5 for Sectore to 1 362 for sub industry which explains that we can't conclude that sub industry is more informative (sampling noise)

- smaller groups ( like sub insudtry ) give as more noiser estimates of missignes rate ... we will use Sector

In [ ]:
# we want to check if the groups responsible of the min and max are tiny groups or large groups

rates= x_train.groupby('SUB_INDUSTRY')[missing_col].mean()
sizes= x_train.groupby('SUB_INDUSTRY').size()

feature="VOLUME_5_missing"
combined=pd.DataFrame(
    {"rate": rates[feature],
     "size": sizes   
     } 
)
combined.sort_values('rate',ascending=False).head(5)



,rate,size
SUB_INDUSTRY,,
20,0.313725,459
36,0.303448,145
135,0.303116,353
116,0.301075,93
125,0.256214,523


- we can see that the max rate come from both large and tiny groups (not only from groups of size near to the median 32 )

- we can conclude that the elvated missingnes in the sub industries is a robust MAR signal 

In [13]:
for col in features_with_mv.index:
    x_train[col]=x_train.groupby('SUB_INDUSTRY')[col].transform(lambda x: x.fillna(x.median()))


In [18]:
# the median of a goupe of sub industries with all NAN values is nan we need to check if there s missing values after the imputation
x_train[features_with_mv.index].isna().sum()[x_train[features_with_mv.index].isna().sum()>0]

Series([], dtype: int64)

- imputation done

In [24]:
x_train.to_csv(DATA_DIR_PROCESSED/"x_train_processed.csv")